# microsoft_agents_hosting_core smoke test

Tests for the `microsoft-agents-hosting-core` package (Microsoft Agents SDK hosting core).

In [ ]:
# Example 1: Top-level public API discovery.
from microsoft_agents.hosting import core as hosting_core

public_names = [name for name in dir(hosting_core) if not name.startswith('_')]
print(f"Public names on microsoft_agents.hosting.core: {len(public_names)}")
for name in public_names[:20]:
    obj = getattr(hosting_core, name)
    print(f"  {name} ({type(obj).__name__})")

assert len(public_names) > 0, "Expected at least one public attribute on the package"

In [ ]:
# Example 2: Import core submodules that are typical for an agent-hosting library.
import importlib

candidate_modules = [
    'microsoft_agents.hosting.core.app',
    'microsoft_agents.hosting.core.authorization',
    'microsoft_agents.hosting.core.storage',
    'microsoft_agents.hosting.core.state',
    'microsoft_agents.hosting.core.turn_context',
]

found = []
for modname in candidate_modules:
    mod = importlib.import_module(modname)
    found.append((modname, len([n for n in dir(mod) if not n.startswith('_')])))

for modname, count in found:
    print(f"  {modname}: {count} public names")

print(f"Total submodules imported: {len(found)}")
# Every one of the five core submodules must import (a missing one fails the cell)
assert [name for name, _ in found] == candidate_modules, found

In [ ]:
# Example 3: Inspect any classes exposed at the top level.
import inspect
from microsoft_agents.hosting import core as hosting_core

classes = []
functions = []
for name in dir(hosting_core):
    if name.startswith('_'):
        continue
    obj = getattr(hosting_core, name)
    if inspect.isclass(obj):
        classes.append(name)
    elif inspect.isfunction(obj):
        functions.append(name)

print(f"Top-level classes ({len(classes)}):")
for cls_name in classes[:15]:
    print(f"  {cls_name}")
print(f"Top-level functions ({len(functions)}):")
for fn_name in functions[:10]:
    print(f"  {fn_name}")

# The classes an agent is built from are exported at the top level.
assert {'ActivityHandler', 'ChannelAdapter', 'TurnContext'} <= set(classes), classes

In [ ]:
# Example 4: Verify the activity dependency is importable (microsoft_agents_activity is a runtime dep).
import importlib

# microsoft_agents.activity is a REQUIRED runtime dependency, so it imports directly (a failure fails the cell)
activity_mod = importlib.import_module('microsoft_agents.activity')
activity_names = [n for n in dir(activity_mod) if not n.startswith('_')]
print(f"microsoft_agents.activity imported: {len(activity_names)} public names")
print(f"Sample: {activity_names[:10]}")
assert activity_names, "microsoft_agents.activity exposes no public names"

from microsoft_agents.activity import Activity
from microsoft_agents.hosting.core import ActivityHandler, ChannelAdapter, TurnContext

class RecordingAdapter(ChannelAdapter):
    async def send_activities(self, context, activities):
        self.sent = [*getattr(self, "sent", []), *activities]
        return []
    async def update_activity(self, context, activity): ...
    async def delete_activity(self, context, reference): ...

class EchoHandler(ActivityHandler):
    async def on_message_activity(self, turn_context):
        await turn_context.send_activity(f"echo {turn_context.activity.text}")

adapter = RecordingAdapter()
# service_url is required by the conversation reference; nothing contacts it (the adapter records instead of posting).
incoming = Activity(type="message", text="hello", channel_id="test", service_url="http://localhost",
                    conversation={"id": "conv-1"}, from_property={"id": "user-1"}, recipient={"id": "bot-1"})
turn = TurnContext(adapter, incoming)
await EchoHandler().on_turn(turn)
# The reply is addressed back to the sender, in the same conversation: TurnContext applied the conversation reference.
replies = [(reply.type, reply.text, reply.conversation.id, reply.recipient.id) for reply in getattr(adapter, "sent", [])]
print(f"replies {replies}")
assert turn.responded and replies == [("message", "echo hello", "conv-1", "user-1")], (turn.responded, replies)

In [ ]:
# Example 5: Final assertion — package can be re-imported via the dotted name + has non-empty path.
import importlib
import sys

# Force a fresh import via importlib to verify module spec resolution works.
hosting_core = importlib.import_module('microsoft_agents.hosting.core')

assert hosting_core.__name__ == 'microsoft_agents.hosting.core'
assert hasattr(hosting_core, '__path__'), "Expected a package (with __path__), not a single-file module"
assert len(list(hosting_core.__path__)) > 0, "__path__ should be non-empty"
assert 'microsoft_agents.hosting.core' in sys.modules

print("microsoft_agents_hosting_core: PASS")